# GardnAIze ML — database inspection
Inspect the independent `gardnaize_ml` PostgreSQL database after Flyway migrations.


In [1]:
import sys
import sklearn
import pandas as pd

print("Python :", sys.version)
print("Interpréteur :", sys.executable)
print("Scikit-learn :", sklearn.__version__)
print("Pandas :", pd.__version__)

Python : 3.13.16 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:24:22) [GCC 14.3.0]
Interpréteur : /home/bsui/miniconda3/envs/gardnaize-ml/bin/python
Scikit-learn : 1.9.1
Pandas : 3.1.0


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

# Locate .env at the repository root
project_root = Path.cwd()
if not (project_root / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

# Build the connection URL from environment variables
database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("DB_USER", "gardnaize"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("ML_DB_HOST", "127.0.0.1"),
    port=int(os.getenv("ML_DB_PORT", "5433")),
    database=os.getenv("ML_DB_NAME", "gardnaize_ml"),
)

# Create SQLAlchemy engine
engine = create_engine(database_url, pool_pre_ping=True)

print("Engine created successfully")

Engine created successfully


In [5]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT current_database(), current_user
    """))
    print(result.fetchone())

('gardnaize', 'gardnaize')


In [7]:
import os
from sqlalchemy import create_engine, text

# Retrieve the database connection URL
load_dotenv()
database_url = os.getenv("GARDNAIZE_ML_DATABASE_URL")

if not database_url:
    raise RuntimeError(
        "GARDNAIZE_ML_DATABASE_URL is not defined"
    )

# Create the SQLAlchemy engine
engine = create_engine(
    database_url,
    pool_pre_ping=True
)

print("SQLAlchemy engine created successfully")

SQLAlchemy engine created successfully


In [18]:
import os
from sqlalchemy import create_engine, URL

database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("DB_USER", "gardnaize"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("ML_DB_HOST", "127.0.0.1"),
    port=int(os.getenv("ML_DB_PORT", "5433")),
    database=os.getenv("ML_DB_NAME", "gardnaize_ml"),
)

# Dispose of the previous connection pool if it exists
if "engine" in globals():
    engine.dispose()

engine = create_engine(
    database_url,
    pool_pre_ping=True
)

print("Engine recreated successfully")

Engine recreated successfully


In [19]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT current_database(), current_user")
    )
    print(result.fetchone())

('gardnaize', 'gardnaize')


In [20]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            current_database() AS database_name,
            current_user AS username,
            current_schema() AS schema_name,
            version() AS postgresql_version
    """))

    for row in result:
        print(row)

('gardnaize', 'gardnaize', 'gardnaize', 'PostgreSQL 18.6 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit')


In [22]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

# Find the project root
project_root = Path.cwd()

if not (project_root / ".env").is_file():
    project_root = project_root.parent

env_file = project_root / ".env"

if not env_file.is_file():
    raise FileNotFoundError(f".env not found: {env_file}")

# Reload the latest environment configuration
load_dotenv(env_file, override=True)

# Build the ML database connection URL
database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("DB_USER", "gardnaize"),
    password=os.getenv("DB_PASSWORD"),
    host="127.0.0.1",
    port=5433,
    database="gardnaize_ml",
)

# Close the previous engine if necessary
if "engine" in globals():
    engine.dispose()

# Create a single engine for the ML database
engine = create_engine(
    database_url,
    pool_pre_ping=True
)

# Verify the actual connection
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            current_database() AS database_name,
            current_user AS username,
            current_schema() AS schema_name
    """))
    print(result.fetchone())

('gardnaize_ml', 'gardnaize', 'public')


In [23]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
url = os.environ.get('GARDNAIZE_ML_DATABASE_URL')
if not url:
    raise RuntimeError('Set GARDNAIZE_ML_DATABASE_URL in the VS Code/Jupyter environment.')

with engine.connect() as conn:
    print('Connected to:', conn.execute(text('SELECT current_database()')).scalar_one())


Connected to: gardnaize_ml


In [24]:
with engine.connect() as conn:
    tables = pd.read_sql(text("SELECT table_name FROM information_schema.tables WHERE table_schema='horticulture' AND table_type='BASE TABLE' ORDER BY table_name"), conn)
display(tables)


,table_name
0,data_source
1,flyway_schema_history
2,import_run
3,ml_experiment
4,ml_garden_plant_assessment
5,ml_garden_sample
6,plant
7,plant_attribute_evidence
8,plant_color
9,plant_color_assignment


In [25]:
with engine.connect() as conn:
    counts = pd.read_sql(text("SELECT 'plant' AS entity, count(*) AS rows FROM horticulture.plant UNION ALL SELECT 'plant_type', count(*) FROM horticulture.plant_type UNION ALL SELECT 'soil_type', count(*) FROM horticulture.soil_type UNION ALL SELECT 'sun_exposure', count(*) FROM horticulture.sun_exposure UNION ALL SELECT 'ml_garden_sample', count(*) FROM horticulture.ml_garden_sample"), conn)
display(counts)


,entity,rows
0,plant,0
1,plant_type,10
2,soil_type,7
3,sun_exposure,3
4,ml_garden_sample,0


In [26]:
with engine.connect() as conn:
    plants = pd.read_sql(text('SELECT plant_id, scientific_name, common_name, quality_status FROM horticulture.plant ORDER BY plant_id LIMIT 20'), conn)
display(plants)


,plant_id,scientific_name,common_name,quality_status
